# Your First Incubator Services

**The notebooks in this folder assume that the incubator DT is already running**.


**If not**:
1. Please skim through the documentation of the incubator digital twin and pay a special attention to how the containers are set up for communication and time series database. You will see that they follow a similar approach to what has been proposed in the [pre-requisites tutorials](../0-Pre-requisites).
2. Make sure there are no docker containers running, to avoid clashes with the containers used by the incubator DT.
3. Follow the instructions in the [incubator_dt repository README](../incubator_dt/README.md) (section `Running the Digital Twin`) to get it started (by means of running the [start_all_services.py](../incubator_dt/software/startup/start_all_services.py) script), with the following adjustments:
   1. If you wish, you can reuse the virtual environment you have possibly already created in the current repository.
   2. Comment out the line `start_as_daemon(start_energy_saver)` to avoid starting the energy saver service, which conflicts with the PT Reconfiguration Service you will create later. The [start_all_services.py](../incubator_dt/software/startup/start_all_services.py) should therefore look like the following:
      ```python
      ...

      if __name__ == '__main__':
         start_docker_rabbitmq()
         start_docker_influxdb()

         start_as_daemon(start_incubator_realtime_mockup)
         start_as_daemon(start_low_level_driver_mockup)
         start_as_daemon(start_influx_data_recorder)
         start_as_daemon(start_plant_kalmanfilter)

         # start_as_daemon(start_plant_simulator)
         # start_as_daemon(start_simulator)
         # start_as_daemon(start_calibrator)

         # Choose one of the controllers below:
         start_as_daemon(start_controller_physical)
         # start_as_daemon(start_controller_physical_open_loop)

         # Enable self adaptation
         # start_as_daemon(start_self_adaptation_manager)
         # start_as_daemon(start_supervisor)
         # start_as_daemon(start_energy_saver)

      ```
   3. **Note** that the initial setup may take some time, as the docker images are being built.

After the incubator DT is running, you should be able to see an output like the following:
````
time           execution_interval  elapsed  heater_on  fan_on   room   box_air_temperature  state 
19/11 16:17:59  3.00                0.01     True       False   10.70  19.68                Heating
19/11 16:18:02  3.00                0.03     True       True    10.70  19.57                Heating
19/11 16:18:05  3.00                0.01     True       True    10.70  19.57                Heating
19/11 16:18:08  3.00                0.01     True       True    10.69  19.47                Heating
19/11 16:18:11  3.00                0.01     True       True    10.69  19.41                Heating
````

If that's the case, continue running this notebook.

## What is a Digital Twin (DT) Service?

In the context of the incubator digital twin, a service is a process that communicates via a [RabbitMQ message exchange](https://www.rabbitmq.com/), running within a [Docker](https://www.docker.com/products/docker-desktop) container. 

### Types of DT Services

Currently, there are two types of services:

1. **Server Services**  
   These services operate similarly to a client-server model. They perform actions only in response to RabbitMQ messages that encapsulate requests for specific computations. Once the computation is completed, the result is sent back as a RabbitMQ message in response to the original request.

2. **Reactive Services**  
   These services subscribe to a particular RabbitMQ topic. When a message arrives on the subscribed topic, they perform a computation and publish the result to another RabbitMQ topic. Other services can then subscribe to the topic where the result is published, allowing a chain of computations across services.

### Example Services

The following examples demonstrate one of each service type, including a mixed service that both acts as a client server, but also has ongoing computation as a reactive service. The examples also make the services communicate with each other and the incubator physical twin. Altogether these services give you everything you need to build your own digital twin services using rabbitmq.

- **AverageService**: This is a server service that computes the average of a given list of values.
- **MovingAverageTemperatureService**: This is a hybrid service that maintains a moving average of the last N values of the temperature in the incubator. It communicates with the average service to get the average. In addition, as a server, it responds to requests to reset the moving average.
- **PTReconfigurationService**: This is a reactive service that reconfigures the incubator physical twin whenever the difference between the temperature moving average and the actual temperature is high. It uses the values of the `MovingAverageTemperatureService`.

The following summarizes the services:

![](./overview_services.svg)

### Configuration

Most services use the configuration file located at the [startup.conf](../incubator_dt/software/startup.conf) of the incubator DT project for parameters such as RabbitMQ connection details and other settings.

## Setting up dependencies

First we're going to set up the PYTHONPATH so that we can load the python code from the [incubator DT repository](../incubator_dt/software/).

In [1]:
# Configure python path to load incubator modules
import sys
import os

# Get the current working directory. Should be 1-Incubator-Service
current_dir = os.getcwd()

assert os.path.basename(current_dir) == '1-Incubator-Service', 'Current directory is not 1-Incubator-Service'

# Get the parent directory. Should be the root of the repository
parent_dir = os.path.dirname(current_dir)

# The root of the repo should contain the incubator_dt folder. Otherwise something went wrong in 0-Pre-requisites.
assert os.path.exists(os.path.join(parent_dir, 'incubator_dt')), 'incubator_dt folder not found in the repository root'

incubator_dt_software_dir = os.path.join(parent_dir, 'incubator_dt', 'software')

assert os.path.exists(incubator_dt_software_dir), 'incubator_dt software directory not found'

# Add the parent directory to sys.path
sys.path.append(incubator_dt_software_dir)

In [2]:
# After the above, we should be able to import incubator modules

# The following imports a class that makes it easier to interact with the RabbitMQ server, just to test the above code.
from incubator.communication.server.rabbitmq import Rabbitmq

## Connection configuration for services

All DT services will use the configuration from the startup.conf file, available inside the incubator repository:

In [3]:
# Print contents of startup.conf file
startup_conf = os.path.join(os.path.dirname(os.getcwd()), 'incubator_dt', 'software','startup.conf')
assert os.path.exists(startup_conf), 'startup.conf file not found'
with open(startup_conf, 'r') as f:
    print(f.read())


# If you change this file, remember to not commit it to the repository.
rabbitmq: {
    ip = "localhost"
    port = 5672
    username = incubator
    password = incubator
    exchange = Incubator_AMQP
    type = topic
    vhost = /
    # ssl: {   # Enable for ssl support. Only works if the RabbitMQ server is configured to support it.
    #     protocol: "PROTOCOL_TLS",
    #     ciphers : "ECDHE+AESGCM:!ECDSA"
    # }
}
influxdb: {
    url = http://localhost:8086
    token = "-g7q1xIvZqY8BA82zC7uMmJS1zeTj61SQjDCY40DkY6IpPBpvna2YoQPdSeENiekgVLMd91xA95smSkhhbtO7Q=="
    org = incubator
    bucket = incubator
}
physical_twin: {
    controller: {
        temperature_desired = 35.0,
        lower_bound = 5.0,
        heating_time = 20.0,
        heating_gap = 30.0
    }
    controller_open_loop: {
        n_samples_period = 40,
        n_samples_heating = 5,
    }
}
digital_twin: {
    models: {
        plant: {
            param4: {
                C_air = 267.55929458,
                G_b